<a href="https://colab.research.google.com/github/franciscogarate/mcaf/blob/master/notebooks/Ejercicio_12_Mejor_Estimacion_Decesos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/franciscogarate/mcaf

In [ ]:
!pip install pyliferisk

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from mcaf import clr

In [ ]:
from pyliferisk import MortalityTable, lx
from pyliferisk.mortalitytables import PASEM2020_Decesos_M_2ord
mt = MortalityTable(qx=PASEM2020_Decesos_M_2ord)

In [ ]:
edad = 50
df = pd.DataFrame(pd.date_range(start='2025-12-31',periods=(mt.w -edad),freq='YE'), columns=['Fecha'])
df['edad'] = edad + df.index
df['t'] = df.index
df['lx'] = df['edad'].apply(lambda x: mt.lx[x+1] if x <= mt.w else 0)
df.head(5)

In [ ]:
df['qx'] = df['lx'].diff(-1).fillna(0)/df['lx'][0]
df['sum_qx'] = df['qx'].cumsum()
df['px'] = 1 - df['sum_qx']
df.head(5)

In [ ]:
def incr_capital(capital, t):
  return capital * (1 + 0.015) ** t

df['capital'] = incr_capital(5000, df.t)
df.head(5)

In [ ]:
df['pagos'] = df['capital'] * df['qx']
df.head(5)

In [ ]:
df['qx'].sum()

In [ ]:
df['caida'] = 0.03
df['polizas'] = (1 - df['caida']).cumprod()
df.head()

In [ ]:
df['clr'] = df['t'].apply(lambda t: clr[t])
df['factor_desc'] = df.apply(lambda x: 1 / (1 + x.clr) ** (x.t), axis=1)
df['polizas_benef'] =  df['px'] * df['polizas']
df.head(2)

In [ ]:
df['pagos_prob'] = df['pagos'] * df['polizas_benef']
df.head(2)

In [ ]:
df['gastos'] = df['capital'] * 0.01
df['gastos_prob'] = df['gastos'] * df['polizas_benef']
df.head(2)

In [ ]:
df['salidas_prob'] = df['pagos_prob'] + df['gastos_prob']
df.head(2)

In [ ]:
df.salidas_prob @ df.factor_desc